# 03 — CI/CD & Security (v0.3)

Demonstra o pipeline de CI/CD (`.github/workflows/*.yml`) e a governança de
segurança configurada na raiz do projeto (checkov, tflint, pre-commit).
Este notebook apenas **faz parsing dos arquivos YAML e explica cada job** —
não dispara nenhum workflow real (isso só acontece no GitHub, ao abrir um
PR).

In [1]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing PROJECT_LOG.md
    is found. Works whether the notebook is executed from notebooks/ (the
    normal case) or from the repo root."""
    p = start.resolve()
    for _ in range(8):
        if (p / "PROJECT_LOG.md").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Could not locate project root (PROJECT_LOG.md not found upward from %s)" % start)

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Caterpillar (Terminar)


In [2]:
import yaml

WORKFLOWS_DIR = PROJECT_ROOT / ".github" / "workflows"
workflow_files = sorted(WORKFLOWS_DIR.glob("*.yml")) if WORKFLOWS_DIR.exists() else []
print(f"{len(workflow_files)} workflow(s) encontrado(s) em {WORKFLOWS_DIR}:")
for f in workflow_files:
    print(f"  - {f.name}")

5 workflow(s) encontrado(s) em G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\Caterpillar (Terminar)\.github\workflows:
  - ansible-lint.yml
  - powershell-tests.yml
  - python-tests.yml
  - security-scan.yml
  - terraform-ci.yml


## Jobs e steps de cada workflow

In [3]:
def summarize_workflow(path):
    with open(path, "r", encoding="utf-8") as fh:
        data = yaml.safe_load(fh)
    name = data.get("name", path.stem)
    # PyYAML parses the bare `on:` key as boolean True in YAML 1.1 - handle both.
    triggers = data.get("on", data.get(True, {}))
    jobs = data.get("jobs", {})
    print(f"\n=== {path.name} — \"{name}\" ===")
    if isinstance(triggers, dict):
        print(f"  triggers: {list(triggers.keys())}")
    else:
        print(f"  triggers: {triggers}")
    for job_id, job_def in jobs.items():
        steps = job_def.get("steps", [])
        step_names = [s.get("name", s.get("uses", s.get("run", "")[:40])) for s in steps]
        runs_on = job_def.get("runs-on", "?")
        condition = job_def.get("if")
        cond_str = f" [if: {condition}]" if condition else ""
        print(f"  job: {job_id} (runs-on={runs_on}){cond_str}")
        for sn in step_names:
            print(f"      - {sn}")

for f in workflow_files:
    summarize_workflow(f)


=== ansible-lint.yml — "Ansible Lint" ===
  triggers: ['pull_request', 'workflow_dispatch']
  job: ansible-lint (runs-on=ubuntu-latest)
      - Checkout
      - Set up Python
      - Install ansible + ansible-lint
      - Run ansible-lint



=== powershell-tests.yml — "PowerShell Tests" ===
  triggers: ['pull_request', 'workflow_dispatch']
  job: pester (runs-on=windows-latest)
      - Checkout
      - Install/Update Pester
      - Run Pester tests (powershell/tests)
      - Upload Pester results
      - Install PSScriptAnalyzer
      - Run PSScriptAnalyzer (powershell/)



=== python-tests.yml — "Python Tests" ===
  triggers: ['pull_request', 'workflow_dispatch']
  job: pytest (runs-on=ubuntu-latest)
      - Checkout
      - Set up Python ${{ matrix.python-version }}
      - Install dependencies
      - Run pytest with coverage



=== security-scan.yml — "Security Scan (Checkov)" ===
  triggers: ['pull_request', 'workflow_dispatch']
  job: checkov (runs-on=ubuntu-latest)
      - Checkout
      - Run Checkov



=== terraform-ci.yml — "Terraform CI" ===
  triggers: ['pull_request', 'workflow_dispatch']
  job: validate (runs-on=ubuntu-latest)
      - Checkout
      - Setup Terraform
      - Terraform fmt (check, recursive)
      - Terraform init (no backend)
      - Terraform validate
  job: tflint (runs-on=ubuntu-latest)
      - Checkout
      - Setup TFLint
      - Init TFLint plugins
      - Run TFLint
  job: plan (runs-on=ubuntu-latest) [if: ${{ secrets.AWS_ROLE_TO_ASSUME != '' }}]
      - Checkout
      - Setup Terraform
      - Configure AWS credentials (OIDC)
      - Terraform init
      - Terraform plan
  job: plan-skipped-notice (runs-on=ubuntu-latest) [if: ${{ secrets.AWS_ROLE_TO_ASSUME == '' }}]
      - Notice


## O que cada pipeline garante

| Workflow | Garante |
|----------|---------|
| `terraform-ci.yml` | `fmt -check` + `validate` + `tflint` sempre; `terraform plan` somente se `AWS_ROLE_TO_ASSUME` estiver configurado (via OIDC) — nunca `apply` |
| `security-scan.yml` | `checkov` contra `terraform/` — HIGH/CRITICAL bloqueiam o PR, MEDIUM/LOW são reportados sem bloquear |
| `ansible-lint.yml` | Lint das roles/playbooks Ansible (roda em runner Ubuntu — não depende de módulos POSIX locais no Windows) |
| `python-tests.yml` | `pytest` do pacote `automation/` (inclui testes com `moto`, sem custo AWS real) |
| `powershell-tests.yml` | Testes Pester do módulo `powershell/modules/InfraOps` |

## `.checkov.yaml` — política de severidade

In [4]:
checkov_cfg_path = PROJECT_ROOT / ".checkov.yaml"
if checkov_cfg_path.exists():
    print(checkov_cfg_path.read_text(encoding="utf-8"))
else:
    print(".checkov.yaml não encontrado.")

# Checkov configuration — Enterprise Cloud Automation & Infrastructure
# Platform.
#
# Policy: MEDIUM/LOW findings are reported but do not block the pipeline
# (soft-fail); HIGH/CRITICAL findings block the PR (hard-fail). checkov's
# hard-fail-on takes precedence over soft-fail-on when a check matches
# both, so this configuration means "warn on everything, but only actually
# fail the build on HIGH or CRITICAL".
#
# Used by .github/workflows/security-scan.yml via `--config-file .checkov.yaml`
# (or the checkov-action `config_file` input, which passes the same flag).

framework:
  - terraform

directory:
  - terraform

soft-fail-on:
  - MEDIUM
  - LOW

hard-fail-on:
  - HIGH
  - CRITICAL

compact: true
quiet: false

# Skip checks intentionally out of scope for this portfolio project.
# Keep empty until a specific, justified exception is needed — add the
# checkov check ID and a one-line reason as a comment, e.g.:
# skip-check:
#   - CKV_AWS_XXX  # reason: ...



## `.tflint.hcl` — regras de lint do Terraform

In [5]:
tflint_cfg_path = PROJECT_ROOT / ".tflint.hcl"
if tflint_cfg_path.exists():
    print(tflint_cfg_path.read_text(encoding="utf-8"))
else:
    print(".tflint.hcl não encontrado.")

# TFLint configuration — Enterprise Cloud Automation & Infrastructure
# Platform.
#
# Used by .github/workflows/terraform-ci.yml (`tflint` job) against each
# environment under terraform/environments/*. Run locally (optional) from
# the repo root with:
#   tflint --init
#   tflint --recursive

plugin "aws" {
  enabled = true
  version = "0.31.0"
  source  = "github.com/terraform-linters/tflint-ruleset-aws"
}

config {
  format  = "compact"
  module  = true
  force   = false
}

rule "terraform_deprecated_interpolation" {
  enabled = true
}

rule "terraform_deprecated_index" {
  enabled = true
}

rule "terraform_unused_declarations" {
  enabled = true
}

rule "terraform_comment_syntax" {
  enabled = true
}

rule "terraform_documented_outputs" {
  enabled = false
}

rule "terraform_documented_variables" {
  enabled = false
}

rule "terraform_typed_variables" {
  enabled = true
}

rule "terraform_naming_convention" {
  enabled = true
  format  = "snake_case"
}

rule "terraform_standard_mod

## `.pre-commit-config.yaml` — checks locais opcionais

In [6]:
precommit_cfg_path = PROJECT_ROOT / ".pre-commit-config.yaml"
if precommit_cfg_path.exists():
    cfg = yaml.safe_load(precommit_cfg_path.read_text(encoding="utf-8"))
    for repo in cfg.get("repos", []):
        repo_url = repo.get("repo")
        hook_ids = [h.get("id") for h in repo.get("hooks", [])]
        print(f"{repo_url}\n  hooks: {hook_ids}")
else:
    print(".pre-commit-config.yaml não encontrado.")

https://github.com/pre-commit/pre-commit-hooks
  hooks: ['end-of-file-fixer', 'trailing-whitespace', 'check-yaml', 'check-merge-conflict', 'check-added-large-files']
https://github.com/antonbabenko/pre-commit-terraform
  hooks: ['terraform_fmt']


## Resumo

- 5 workflows do GitHub Actions foram parseados diretamente do YAML —
  jobs, steps, condições de execução (`if`) e triggers exibidos acima.
- A política de segurança (`checkov`) é soft-fail em MEDIUM/LOW e hard-fail
  em HIGH/CRITICAL — reportado explicitamente, nunca bloqueia o pipeline
  por achados de baixo risco.
- `pre-commit` é uma camada **opcional** local; a validação que realmente
  importa (a que bloqueia PRs) roda sempre no GitHub Actions.